In [0]:
passengers_day1 = [
    (101,"Rahul Sharma","Hyderabad","Economy","India"),
    (102,"Priya Reddy","Bangalore","Business","India"),
    (103,"Amit Kumar","Mumbai","Economy","India"),
    (104,"Sneha Patel","Delhi","Premium Economy","India"),
    (105,"Farhan Ali","Chennai","Economy","India")
    ]
columns = [
    "passenger_id",
    "passenger_name",
    "city",
    "travel_class",
    "country"
]

df_day1 = spark.createDataFrame(
    passengers_day1,
    columns
)

In [0]:
passengers_day2 = [
    (102,"Priya Reddy","Bangalore","First Class","India"),
    (104,"Sneha Patel","Hyderabad","Premium Economy","India"),
    (106,"Neha Singh","Pune","Economy","India"),
    (107,"Arjun Verma","Kochi","Business","India")
]

df_day2 = spark.createDataFrame(
    passengers_day2,
    columns
)

In [0]:
df_day1.write.format("delta").mode("overwrite") \
    .saveAsTable("passengers_delta")

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(
    spark,
    "passengers_delta"
)

delta_table.alias("target").merge(
    df_day2.alias("source"),
    "target.passenger_id = source.passenger_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
select * from passengers_delta

passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
103,Amit Kumar,Mumbai,Economy,India
105,Farhan Ali,Chennai,Economy,India
102,Priya Reddy,Bangalore,First Class,India
104,Sneha Patel,Hyderabad,Premium Economy,India
106,Neha Singh,Pune,Economy,India
107,Arjun Verma,Kochi,Business,India


In [0]:
print('day 01 count: ', df_day1.count())
print('day 02 count: ', df_day2.count())
print("Record Count:", spark.table("passengers_delta").count())

day 01 count:  5
day 02 count:  4
Record Count: 7


In [0]:
%sql
select * from passengers_delta

passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
103,Amit Kumar,Mumbai,Economy,India
105,Farhan Ali,Chennai,Economy,India
102,Priya Reddy,Bangalore,First Class,India
104,Sneha Patel,Hyderabad,Premium Economy,India
106,Neha Singh,Pune,Economy,India
107,Arjun Verma,Kochi,Business,India


In [0]:
%sql DESCRIBE HISTORY passengers_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-06-17T11:30:42.000Z,146722045516591,azuser7218_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2380851102976800),1ba4a422-c295-4ccb-8f20-c05eca4beabb,0617-093035-kghxfbj-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 9126, p25FileSize -> 2047, numDeletionVectorsRemoved -> 1, minFileSize -> 2047, numAddedFiles -> 1, maxFileSize -> 2047, p75FileSize -> 2047, p50FileSize -> 2047, numAddedBytes -> 2047)",null,Databricks-Runtime/18.2.x-photon-scala2.13
1,2026-06-17T11:30:40.000Z,146722045516591,azuser7218_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(passenger_id#16123L = passenger_id#16148L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2380851102976800),1ba4a422-c295-4ccb-8f20-c05eca4beabb,0617-093035-kghxfbj-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 7131, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 3726, materializeSourceTimeMs -> 96, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1753, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1839)",null,Databricks-Runtime/18.2.x-photon-scala2.13
0,2026-06-17T11:29:52.000Z,146722045516591,azuser7218_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2380851102976800),5a06c32b-c470-47a5-a1cf-4b010294225f,0617-093035-kghxfbj-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 1995)",null,Databricks-Runtime/18.2.x-photon-scala2.13


In [0]:
%sql
update passengers_delta
set travel_class = 'Premium Economy'
where passenger_id = 102;

insert into passengers_delta
values(
    131,
    'John',
    'New York',
    'First Class',
    'USA'
);

select passenger_id, passenger_name, travel_class from passengers_delta
where passenger_id = 102;

passenger_id,passenger_name,travel_class
102,Priya Reddy,Premium Economy


In [0]:
spark.read.format('delta').option('versionAsOf', 0).table('passengers_delta').filter('passenger_id = 102').show()

spark.read.format('delta').option('versionAsOf', 1).table('passengers_delta').filter('passenger_id = 102').show()

+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore|    Business|  India|
+------------+--------------+---------+------------+-------+

+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore| First Class|  India|
+------------+--------------+---------+------------+-------+



In [0]:
spark.read.format('delta').option('versionAsOf', 0).table('passengers_delta').filter('passenger_id = 106').show()

spark.read.format('delta').option('versionAsOf', 1).table('passengers_delta').filter('passenger_id = 106').show()

+------------+--------------+----+------------+-------+
|passenger_id|passenger_name|city|travel_class|country|
+------------+--------------+----+------------+-------+
+------------+--------------+----+------------+-------+

+------------+--------------+----+------------+-------+
|passenger_id|passenger_name|city|travel_class|country|
+------------+--------------+----+------------+-------+
|         106|    Neha Singh|Pune|     Economy|  India|
+------------+--------------+----+------------+-------+



In [0]:
%sql describe history passengers_delta

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-06-17T11:48:55.000Z,146722045516591,azuser7218_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2380851102976800),7102644d-2892-4185-b3d6-fa0bdbb7d999,0617-093035-kghxfbj-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 3867, p25FileSize -> 2032, numDeletionVectorsRemoved -> 1, conflictDetectionTimeMs -> 79, minFileSize -> 2032, numAddedFiles -> 1, maxFileSize -> 2032, p75FileSize -> 2032, p50FileSize -> 2032, numAddedBytes -> 2032)",null,Databricks-Runtime/18.2.x-photon-scala2.13
4,2026-06-17T11:48:53.000Z,146722045516591,azuser7218_mml.local@karthikirisoutlook.onmicrosoft.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2380851102976800),d85c1b69-4e9b-40fb-871e-86f1ef2164c4,0617-093035-kghxfbj-v2n,3,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1715)",null,Databricks-Runtime/18.2.x-photon-scala2.13
3,2026-06-17T11:48:52.000Z,146722045516591,azuser7218_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(passenger_id#17365L = 102)""])",null,List(2380851102976800),7102644d-2892-4185-b3d6-fa0bdbb7d999,0617-093035-kghxfbj-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2717, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1438, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1820, rewriteTimeMs -> 1273)",null,Databricks-Runtime/18.2.x-photon-scala2.13
2,2026-06-17T11:30:42.000Z,146722045516591,azuser7218_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2380851102976800),1ba4a422-c295-4ccb-8f20-c05eca4beabb,0617-093035-kghxfbj-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 9126, p25FileSize -> 2047, numDeletionVectorsRemoved -> 1, minFileSize -> 2047, numAddedFiles -> 1, maxFileSize -> 2047, p75FileSize -> 2047, p50FileSize -> 2047, numAddedBytes -> 2047)",null,Databricks-Runtime/18.2.x-photon-scala2.13
1,2026-06-17T11:30:40.000Z,146722045516591,azuser7218_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(passenger_id#16123L = passenger_id#16148L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2380851102976800),1ba4a422-c295-4ccb-8f20-c05eca4beabb,0617-093035-kghxfbj-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 7131, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 3726, materializeSourceTimeMs -> 96, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1753, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1839)",null,Databricks-Runtime/18.2.x-photon-scala2.13
0,2026-06-17T11:29:52.000Z,146722045516591,azuser7218_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parq

In [0]:
v1_df = spark.read.format('delta').option('versionAsOf', 0).table('passengers_delta')
v2_df = spark.read.format('delta').table('passengers_delta')

print("Version 0:", v1_df.count())
print("Version 1:", v2_df.count())

v1_df.filter('passenger_id = 102').show()
v2_df.filter('passenger_id = 102').show()

v1_df.filter('passenger_id = 104').show()
v2_df.filter('passenger_id = 104').show()

Version 0: 5
Version 1: 8
+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore|    Business|  India|
+------------+--------------+---------+------------+-------+

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         102|   Priya Reddy|Bangalore|Premium Economy|  India|
+------------+--------------+---------+---------------+-------+

+------------+--------------+-----+---------------+-------+
|passenger_id|passenger_name| city|   travel_class|country|
+------------+--------------+-----+---------------+-------+
|         104|   Sneha Patel|Delhi|Premium Economy|  India|
+------------+--------------+-----+---------------+-------+

+------------+--------------+---------+-------

In [0]:
%sql
optimize passengers_delta;

optimize passengers_delta
zorder by city;

delete from passengers_delta 
where passenger_id = 105;

vacuum passengers_delta;

describe history passengers_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
13,2026-06-17T12:06:03.000Z,146722045516591,azuser7218_mml.local@karthikirisoutlook.onmicrosoft.com,VACUUM END,Map(status -> COMPLETED),null,List(2380851102976800),56b4cc12-355a-4326-aa26-6e8af33d88b5,0617-093035-kghxfbj-v2n,12,SnapshotIsolation,true,"Map(numDeletedFiles -> 0, numVacuumedDirectories -> 1)",null,Databricks-Runtime/18.2.x-photon-scala2.13
12,2026-06-17T12:06:02.000Z,146722045516591,azuser7218_mml.local@karthikirisoutlook.onmicrosoft.com,VACUUM START,"Map(retentionCheckEnabled -> true, defaultRetentionMillis -> 604800000)",null,List(2380851102976800),56b4cc12-355a-4326-aa26-6e8af33d88b5,0617-093035-kghxfbj-v2n,11,SnapshotIsolation,true,"Map(numFilesToDelete -> 0, sizeOfDataToDelete -> 0)",null,Databricks-Runtime/18.2.x-photon-scala2.13
11,2026-06-17T12:05:58.000Z,146722045516591,azuser7218_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(passenger_id#20842L = 105)""])",null,List(2380851102976800),a09257fc-068d-47ed-9136-c781e35cdbc7,0617-093035-kghxfbj-v2n,10,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 685, numDeletionVectorsUpdated -> 0, numDeletedRows -> 0, scanTimeMs -> 681, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 0)",null,Databricks-Runtime/18.2.x-photon-scala2.13
10,2026-06-17T12:03:46.000Z,146722045516591,azuser7218_mml.local@karthikirisoutlook.onmicrosoft.com,VACUUM END,Map(status -> COMPLETED),null,List(2380851102976800),358586ba-1907-4edd-8a30-abbf51f53eed,0617-093035-kghxfbj-v2n,9,SnapshotIsolation,true,"Map(numDeletedFiles -> 0, numVacuumedDirectories -> 1)",null,Databricks-Runtime/18.2.x-photon-scala2.13
9,2026-06-17T12:03:45.000Z,146722045516591,azuser7218_mml.local@karthikirisoutlook.onmicrosoft.com,VACUUM START,"Map(retentionCheckEnabled -> true, defaultRetentionMillis -> 604800000)",null,List(2380851102976800),358586ba-1907-4edd-8a30-abbf51f53eed,0617-093035-kghxfbj-v2n,8,SnapshotIsolation,true,"Map(numFilesToDelete -> 0, sizeOfDataToDelete -> 0)",null,Databricks-Runtime/18.2.x-photon-scala2.13
8,2026-06-17T12:03:40.000Z,146722045516591,azuser7218_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2380851102976800),0218b8b7-691e-40d0-8ad2-49a972d6e0bb,0617-093035-kghxfbj-v2n,7,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2071, p25FileSize -> 2048, numDeletionVectorsRemoved -> 1, minFileSize -> 2048, numAddedFiles -> 1, maxFileSize -> 2048, p75FileSize -> 2048, p50FileSize -> 2048, numAddedBytes -> 2048)",null,Databricks-Runtime/18.2.x-photon-scala2.13
7,2026-06-17T12:03:39.000Z,146722045516591,azuser7218_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(passenger_id#19020L = 105)""])",null,List(2380851102976800),0218b8b7-691e-40d0-8ad2-49a972d6e0bb,0617-093035-kghxfbj-v2n,6,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1294, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 925, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 367)",null,Databricks-Runtime/18.2.x-photon-scala2.13
6,2026-06-17T12:03:35.000Z,146722045516591,azuser7218_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2380851102976800),8717543b-fdbb-41cc-85ba-335414ef3324,0617-093035-kghxfbj-v2n,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 3747, p25FileSize -> 2071, numDeletionVectorsRemoved -> 0, minFileSiz